# Whirlpool AI Operations Hub: Multi-Agent & BigQuery Vector Search
### Demonstração Prática de IA Generativa, Governança e Busca Vetorial no Google Cloud

Este notebook demonstra o fluxo completo da solução desenvolvida sob medida para a vaga de **AI Analyst** da **Whirlpool**:
1. **Conexão com Google Cloud (Vertex AI e BigQuery)**
2. **Governança de Dados (Compliance PULSE / PIA / LGPD)**
3. **Segmentação e Ingestão Multimodal com Gemini no Vertex AI**
4. **Geração de Embeddings (`text-embedding-004`) e BigQuery Vector Search**
5. **Consultas RAG de Alto Impacto para Tomada de Decisão**
6. **Geração Automatizada de Diagramas de Processos (Mermaid.js)**

## 1. Inicialização e Teste de Conexões

In [1]:
import sys
sys.path.append("..")

from src.utils.gcp_client import test_connections, PROJECT_ID, LOCATION
print(f"Projeto Ativo: {PROJECT_ID} ({LOCATION})")
test_connections()

Projeto Ativo: project-9940e307-bf71-45ef-be0 (us-central1)
Testando conexões no projeto: project-9940e307-bf71-45ef-be0 (us-central1)
[OK] BigQuery conectado com sucesso! Datasets encontrados: 1


Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


[OK] Vertex AI (gemini-2.5-flash) conectado com sucesso!
     Resposta de teste: Melhorar a vida em casa.
Todos os serviços Google Cloud estão operacionais!


True

## 2. Governança de Dados: Mascaramento de PII (Compliance PULSE / PIA)
Antes de qualquer persistência no BigQuery, o `GovernanceAgent` audita o texto e mascara CPFs, matrículas funcionais e dados sensíveis.

In [2]:
from data.sample_meetings import SAMPLE_MEETINGS
from src.agents.governance_agent import GovernanceAgent

sample = SAMPLE_MEETINGS[0]
print("--- TEXTO ORIGINAL (COM PII) ---")
print(sample["raw_text"][:350] + "...")

gov = GovernanceAgent()
result = gov.sanitize_transcript(sample["raw_text"])

print("\n--- AUDITORIA DE GOVERNANÇA ---")
print(f"Métricas: {result['metrics']}")
print(f"Status: {result['compliance_status']}")

print("\n--- TEXTO SANITIZADO ---")
print(result["sanitized_text"][:350] + "...")

--- TEXTO ORIGINAL (COM PII) ---

Participantes: Carlos Silva (Gerente de Logística), Mariana Mendes (Supervisora de Suprimentos), Roberto Souza (Engenharia de Produção).

[00:01] Carlos Silva: Bom dia equipe. Convoquei este alinhamento de emergência devido ao atraso de 14 dias no lote de compressores inverter para a planta de Rio Claro. O fornecedor principal reportou parada técn...

--- AUDITORIA DE GOVERNANÇA ---
Métricas: {'cpf_redactions': 1, 'phone_redactions': 1, 'badge_redactions': 1, 'high_value_redactions': 0}
Status: APPROVED_PULSE_PIA

--- TEXTO SANITIZADO ---
Participantes: Carlos Silva (Gerente de Logística), Mariana Mendes (Supervisora de Suprimentos), Roberto Souza (Engenharia de Produção).

[00:01] Carlos Silva: Bom dia equipe. Convoquei este alinhamento de emergência devido ao atraso de 14 dias no lote de compressores inverter para a planta de Rio Claro. O fornecedor principal reportou parada técni...


## 3. Ingestão e Estruturação Multimodal (Vertex AI)
O `MultimodalMeetingAgent` analisa a discussão e divide o diálogo em chunks semânticos com participantes, tópicos e ações.

In [3]:
from src.agents.multimodal_agent import MultimodalMeetingAgent

agent = MultimodalMeetingAgent()
structured = agent.process_transcript_text(
    meeting_id=sample["meeting_id"],
    title=sample["meeting_title"],
    raw_text=result["sanitized_text"],
    department=sample["department"],
    date=sample["meeting_date"]
)

print(f"Reunião: {structured['meeting_title']}")
print(f"Resumo Executivo: {structured.get('summary')}")
print(f"Decisões Principais: {structured.get('key_decisions')}")
print(f"Total de Chunks gerados: {len(structured.get('chunks', []))}")

Reunião: Alinhamento Operacional: Cadeia de Suprimentos e Compressores - Linha Brastemp
Resumo Executivo: A reunião de emergência abordou o atraso de 14 dias na entrega de compressores inverter para a planta de Rio Claro, ameaçando a paralisação da linha de montagem Brastemp Duplex. Uma solução alternativa foi encontrada utilizando 800 compressores de segunda geração do armazém de Joinville, evitando um custoso frete aéreo e garantindo a continuidade da produção.
Decisões Principais: ['Transferência imediata de 800 compressores de segunda geração do armazém de Joinville para Rio Claro.', 'Aprovação do frete rodoviário dedicado no valor de R$ 68.000,00.', 'Cancelamento da opção de frete aéreo emergencial de R$ 4.850.000,00.']
Total de Chunks gerados: 5


## 4. Geração de Embeddings e BigQuery Vector Search
Executamos uma busca semântica direta no BigQuery usando a função `VECTOR_SEARCH` e similaridade de cosseno.

In [4]:
from src.pipeline.embeddings import generate_query_embedding
from src.pipeline.bigquery_loader import search_similar_chunks

pergunta = "Qual alternativa foi encontrada para os compressores de Rio Claro e quanto custará o frete?"
query_vector = generate_query_embedding(pergunta)

hits = search_similar_chunks(query_vector, top_k=3)
print(f"Resultados recuperados do BigQuery para a pergunta: '{pergunta}'\n")
for i, hit in enumerate(hits, 1):
    print(f"[{i}] Similaridade: {hit['similarity_score']} | Falante: {hit['speaker']} | Dept: {hit['department']}")
    print(f"    Conteúdo: {hit['content']}")
    print("-" * 70)

Resultados recuperados do BigQuery para a pergunta: 'Qual alternativa foi encontrada para os compressores de Rio Claro e quanto custará o frete?'

[1] Similaridade: 0.6564 | Falante: Roberto Souza | Dept: Logística e Manufatura
    Conteúdo: Da parte da Engenharia, temos 800 unidades de compressores de segunda geração no armazém de Joinville que podem ser transferidos via rodoviário expresso. Eles têm compatibilidade mecânica total e passam pelo protocolo de testes PULSE-ENG-402.
----------------------------------------------------------------------
[2] Similaridade: 0.6564 | Falante: Roberto Souza | Dept: Logística e Manufatura
    Conteúdo: Da parte da Engenharia, temos 800 unidades de compressores de segunda geração no armazém de Joinville que podem ser transferidos via rodoviário expresso. Eles têm compatibilidade mecânica total e passam pelo protocolo de testes PULSE-ENG-402.
----------------------------------------------------------------------
[3] Similaridade: 0.6413 | Falante:

## 5. Agente Consultor RAG (Síntese Executiva)
O `RagConsultantAgent` sintetiza a resposta final com embasamento total nos trechos recuperados do BigQuery.

In [5]:
from src.agents.rag_agent import RagConsultantAgent

rag = RagConsultantAgent()
resposta = rag.answer_query("O que foi decidido sobre os compressores de Rio Claro e qual o impacto financeiro?")

print("RESPOSTA DO AGENTE RAG:")
print(resposta["answer"])
print("\nFONTES AUDITADAS:")
for s in resposta["sources"]:
    print(f"- {s['meeting_title']} (Falante: {s['speaker']} | Score: {s['similarity_score']})")


[RAG] Processando pergunta: 'O que foi decidido sobre os compressores de Rio Claro e qual o impacto financeiro?'
RESPOSTA DO AGENTE RAG:
Conforme Carlos Silva em 15/08/2026, foi decidido:
1.  Mariana Mendes deve acionar a transferência imediata de 800 compressores de Joinville para Rio Claro até amanhã às 18h.
2.  Roberto Souza deve preparar o laudo técnico da engenharia para o controle de qualidade.
3.  O impacto financeiro é a aprovação de um frete rodoviário dedicado no valor contingenciado de R$ 68.000,00, cancelando a opção aérea, conforme Carlos Silva.

FONTES AUDITADAS:
- Alinhamento Operacional: Cadeia de Suprimentos e Compressores - Linha Brastemp (Falante: Carlos Silva | Score: 0.6201)
- Alinhamento Operacional: Cadeia de Suprimentos e Compressores - Linha Brastemp (Falante: Carlos Silva | Score: 0.5874)
- Alinhamento Operacional: Cadeia de Suprimentos e Compressores - Linha Brastemp (Falante: Carlos Silva | Score: 0.585)


## 6. Geração de Entregáveis de Processo: Diagramas Mermaid & Matriz RACI
O `ProcessDiagramAgent` mapeia o fluxo operacional da decisão e estrutura a matriz de governança de papéis.

In [6]:
from src.agents.diagram_agent import ProcessDiagramAgent

diagram_agent = ProcessDiagramAgent()
mermaid_code = diagram_agent.generate_operational_diagram(structured)
raci_table = diagram_agent.generate_raci_matrix(structured)

print("--- DIAGRAMA MERMAID GERADO ---")
print(mermaid_code)
print("\n--- MATRIZ RACI ---")
print(raci_table)

--- DIAGRAMA MERMAID GERADO ---
```mermaid
graph TD
    A[Gatilho: Atraso de 14 dias na entrega de Compressores Inverter] --> B[Risco: Paralisação da Linha de Montagem Brastemp Duplex em Rio Claro]

    B --> C{Avaliar Opções de Contingência Urgentes}
    C -- Opção de Alto Custo --> D[Opção 1: Frete Aéreo Emergencial (R$ 4.850.000,00)]
    C -- Opção Alternativa Viável --> E[Opção 2: Utilizar 800 Compressores 2ª Geração do Armazém de Joinville]

    E -- Requer Transporte --> F[Meio de Transporte: Frete Rodoviário Dedicado (R$ 68.000,00)]

    D & F --> G{Decisão Final da Reunião: Qual Solução Implementar?}

    G -- Aprovado --> H[Decisão: Transferência Imediata de 800 Compressores Joinville -> Rio Claro]
    G -- Aprovado --> I[Decisão: Aprovação do Frete Rodoviário Dedicado (R$ 68.000,00)]
    G -- Cancelado --> J[Decisão: Frete Aéreo Emergencial Cancelado (R$ 4.850.000,00)]

    H & I & J --> K(Ação: Acionar Transferência Imediata de Compressores - Mariana Mendes)
    K --> L(Ação